# Data Cleaning Pipeline

**DS2500 — Programming with Data | Northeastern University**  
**Team:** Sebastian Brookes, Shiven Mishra

This notebook walks through the cleaning pipeline for the four raw data sources
used in our election prediction analysis: **Polymarket** (prediction market odds),
**FiveThirtyEight** (polling averages), **FEC** (official results), and a manually
curated **events timeline**. Each section loads the raw data, demonstrates the
cleaning transformations, and inspects the output. The cleaning logic itself lives
in `src/clean/`. This notebook calls those functions and walks through the process.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Add the project root to sys.path so we can import from src/
PROJECT_ROOT = str(Path.cwd().parent)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.clean.utils import (
    POLYMARKET_RAW, FIVETHIRTYEIGHT_RAW, FEC_RAW, EVENTS_RAW,
    PROCESSED_DIR, SWING_STATES,
)
from src.clean.clean_polymarket import main as clean_polymarket
from src.clean.clean_fivethirtyeight import main as clean_fivethirtyeight
from src.clean.clean_fec import main as clean_fec
from src.clean.clean_events import main as clean_events

## Methodology

### Data sources

| Source | Raw format | Granularity | Output file(s) |
| --- | --- | --- | --- |
| **Polymarket** | ~50 per-state CSVs × 3 granularities | State × day/hour/minute | `polymarket_daily.csv`, `_hourly.csv`, `_minutely.csv` |
| **FiveThirtyEight** | 27k-row CSV (2020 + 2024 cycles) | State × day | `polls_538.csv` |
| **FEC** | Excel file with official results | State-level totals | `fec_results.csv` |
| **Events** | Manually curated CSV | Per-event | `events.csv` |

### Pipeline architecture

Each data source has a dedicated cleaning module in `src/clean/` with a `main()`
function that reads from `data/raw/`, applies transformations, and writes to
`data/processed/`. Shared constants (state mappings, swing states, file paths)
live in `src/clean/utils.py`. This notebook calls each `main()` function in
sequence and inspects the results.

---

## Section 1: Polymarket

Polymarket is a prediction market where traders buy and sell shares on election
outcomes. The raw data consists of ~50 per-state CSV files at three time
granularities (daily, hourly, minutely), each recording Trump, Harris, and Other
win-probability columns.

**Cleaning challenges:**
- 150 separate files must be consolidated into 3 output CSVs
- Raw date format (`MM-DD-YYYY HH:MM`) needs parsing
- State markets launched on different dates (staggered rollout)
- Daily data can have multiple timestamps mapping to the same calendar date
- Probabilities should sum to 1.0 (sanity check)

In [ ]:
# ---------------------------------------------------------------------------
# Assert raw directories exist; show file inventory
# ---------------------------------------------------------------------------

for subdir in ["csv_day", "csv_hour", "csv_minute"]:
    path = POLYMARKET_RAW / subdir
    assert path.exists(), f"Missing: {path}"

day_files = sorted((POLYMARKET_RAW / "csv_day").glob("*.csv"))
hour_files = sorted((POLYMARKET_RAW / "csv_hour").glob("*.csv"))
min_files = sorted((POLYMARKET_RAW / "csv_minute").glob("*.csv"))

print(f"Daily files:    {len(day_files)}")
print(f"Hourly files:   {len(hour_files)}")
print(f"Minutely files: {len(min_files)}")
print(f"Total files:    {len(day_files) + len(hour_files) + len(min_files)}")

# Load one sample file to show raw column structure
sample_raw = pd.read_csv(day_files[0])
print(f"\nSample file: {day_files[0].name}")
print(f"Shape: {sample_raw.shape}")
print(f"Columns: {list(sample_raw.columns)}")
print(f"Data types:\n{sample_raw.dtypes}")
display(sample_raw.head(3))

In [ ]:
# ---------------------------------------------------------------------------
# Polymarket didn't launch all state markets simultaneously. The earliest dates
# vary by state, which matters for comparing multiple sources.
# ---------------------------------------------------------------------------

sample_states = ["PA", "AK", "GA", "WI", "TX"]

print("Earliest date per state (daily files):")
for state in sample_states:
    csv_path = POLYMARKET_RAW / "csv_day" / f"{state}_daily.csv"
    df_tmp = pd.read_csv(csv_path)
    raw_dates = df_tmp["Date (UTC)"]
    print(f"  {state}: {raw_dates.iloc[0]}  (raw format, {len(df_tmp)} rows)")

In [ ]:
# ---------------------------------------------------------------------------
# Run the cleaning function
# ---------------------------------------------------------------------------
# Note: the cleaning step below warns about rows where trump_prob + harris_prob
# + other_prob deviates from 1.0 by >0.05. This is normal for prediction
# markets, where prices are set independently and don't always sum to exactly 1.

# Count total raw rows across all daily files for the before/after comparison
raw_daily_rows = sum(len(pd.read_csv(f)) for f in day_files)

clean_polymarket()

# Load the cleaned daily output for inspection
pm_daily = pd.read_csv(PROCESSED_DIR / "polymarket_daily.csv", parse_dates=["date"])

print(f"\nBefore/after (daily):")
print(f"  Raw:     {raw_daily_rows:,} total rows across {len(day_files)} files")
print(f"  Cleaned: {len(pm_daily):,} rows in 1 consolidated CSV")

In [ ]:
# ---------------------------------------------------------------------------
# Inspect the cleaned daily output
# ---------------------------------------------------------------------------

print(f"Shape:      {pm_daily.shape}")
print(f"Date range: {pm_daily['date'].min().date()} to {pm_daily['date'].max().date()}")
print(f"States:     {pm_daily['state'].nunique()}")
print(f"Swing:      {pm_daily[pm_daily['is_swing']]['state'].nunique()} "
      f"({', '.join(sorted(pm_daily[pm_daily['is_swing']]['state'].unique()))})")

# Probability sum sanity check
prob_sum = pm_daily["trump_prob"] + pm_daily["harris_prob"] + pm_daily["other_prob"]
print(f"\nProbability sum check:")
print(f"  Mean: {prob_sum.mean():.4f}")
print(f"  Min:  {prob_sum.min():.4f}")
print(f"  Max:  {prob_sum.max():.4f}")

display(pm_daily.head())

### Polymarket cleaning summary

150 per-state CSV files (50 states × 3 granularities) were consolidated into
three output CSVs. Key transformations: raw `MM-DD-YYYY HH:MM` dates parsed to
proper timestamps, candidate columns renamed to `trump_prob`/`harris_prob`/`other_prob`,
and derived columns added (`trump_lead`, `state_name`, `is_swing`). For daily
data, duplicate timestamps mapping to the same calendar date were resolved by
keeping the latest snapshot. Probabilities were validated to sum to ~1.0.

---

## Section 2: FiveThirtyEight

FiveThirtyEight publishes rolling polling averages for each candidate by state.
The raw file contains ~27,000 rows in **long format** (one row per candidate per
state per date), covering both the 2020 and 2024 election cycles.

**Cleaning challenges:**
- Filter to 2024 cycle only
- Convert long format to wide (one row per date × state)
- Merge Biden and Harris into a single Democratic candidate column
- Not all states have data on every date (irregular polling cadence)

In [ ]:
# ---------------------------------------------------------------------------
# Check raw file exists and inspect raw data
# ---------------------------------------------------------------------------

raw_538_path = (FIVETHIRTYEIGHT_RAW
                / "presidential_general_averages_2024-09-12_uncorrected.csv")
assert raw_538_path.exists(), f"Missing: {raw_538_path}"

raw_538 = pd.read_csv(raw_538_path)
print(f"Shape: {raw_538.shape}")
print(f"Columns: {list(raw_538.columns)}")
print(f"Cycles: {sorted(raw_538['cycle'].unique())}")
print(f"Candidates (2024): {sorted(raw_538[raw_538['cycle'] == 2024]['candidate'].unique())}")

# Show long-format example for a single state
pa_sample = raw_538[(raw_538['state'] == 'Pennsylvania') &
                    (raw_538['cycle'] == 2024)].head(6)
print(f"\nLong-format example (Pennsylvania, 2024):")
display(pa_sample)

### Biden-to-Harris transition

The raw data contains polling averages for both Biden and Harris as separate
candidates. Since Biden dropped out on July 21 and Harris became the Democratic
nominee, the cleaning pipeline merges both into a single `dem_pct` column
with a `dem_candidate` field tracking which candidate (`"Biden"` or `"Harris"`)
the row refers to.

In [ ]:
# ---------------------------------------------------------------------------
# Run the cleaning function
# ---------------------------------------------------------------------------

raw_538_total = len(raw_538)
raw_538_2024 = len(raw_538[raw_538["cycle"] == 2024])

clean_fivethirtyeight()

p538 = pd.read_csv(PROCESSED_DIR / "polls_538.csv", parse_dates=["date"])

print(f"\nBefore/after:")
print(f"  Raw:     {raw_538_total:,} total rows ({raw_538_2024:,} from 2024 cycle)")
print(f"  Cleaned: {len(p538):,} rows (wide format, one row per date × state, 2024 only)")

In [ ]:
# ---------------------------------------------------------------------------
# Inspect cleaned output
# ---------------------------------------------------------------------------

print(f"Shape:      {p538.shape}")
print(f"Date range: {p538['date'].min().date()} to {p538['date'].max().date()}")
print(f"States:     {p538['state'].nunique()} (incl. US national)")

# Biden-Harris transition dates
biden_rows = p538[p538["dem_candidate"] == "Biden"]
harris_rows = p538[p538["dem_candidate"] == "Harris"]
print(f"\nBiden-Harris transition:")
print(f"  Last Biden row:   {biden_rows['date'].max().date()}")
print(f"  First Harris row: {harris_rows['date'].min().date()}")

display(p538.head())

### FiveThirtyEight cleaning summary

The raw long-format file was filtered to the 2024 cycle, pivoted to wide format
(one row per date × state), and Biden/Harris rows were merged into a unified
Democratic candidate column. State names were mapped to two-letter abbreviations
(`"National"` → `"US"`), and derived columns added (`trump_lead`, `state_name`,
`is_swing`). The `pct_trend_adjusted` column (all-null for 2024) and `cycle`
column were dropped.

---

## Section 3: FEC

The Federal Election Commission publishes official, certified election results.
The raw file is an Excel workbook containing vote counts and electoral votes
by state.

**Cleaning challenges:**
- Excel format with a specific sheet name
- Includes rows for U.S. territories and summary totals that must be filtered out
- Vote and electoral-vote columns contain NaN values (for minor candidates)
- Need to derive vote shares, margins, and winner from raw counts

In [ ]:
# ---------------------------------------------------------------------------
# Assert raw file exists and inspect raw data and show what gets filtered out
# ---------------------------------------------------------------------------

raw_fec_path = FEC_RAW / "fec-results.xlsx"
assert raw_fec_path.exists(), f"Missing: {raw_fec_path}"

raw_fec = pd.read_excel(raw_fec_path,
                        sheet_name="OFFICIAL 2024 PRES GE RESULTS",
                        engine="openpyxl")

print(f"Shape: {raw_fec.shape}")
print(f"Columns: {list(raw_fec.columns)}")

# Show which STATE values get filtered out
from src.clean.utils import STATE_ABBREV_TO_NAME
valid_states = set(STATE_ABBREV_TO_NAME.keys())
all_state_vals = set(raw_fec["STATE"].dropna().unique())
filtered_out = sorted(all_state_vals - valid_states)

print(f"\nTotal STATE values in raw data: {len(all_state_vals)}")
print(f"Valid states (50 + DC):         {len(valid_states)}")
print(f"Filtered out ({len(filtered_out)}):             {filtered_out}")

In [ ]:
# ---------------------------------------------------------------------------
# Run the cleaning function
# ---------------------------------------------------------------------------

raw_fec_rows = len(raw_fec)

clean_fec()

fec = pd.read_csv(PROCESSED_DIR / "fec_results.csv")

print(f"\nBefore/after:")
print(f"  Raw:     {raw_fec_rows} rows (incl. territories and totals)")
print(f"  Cleaned: {len(fec)} rows (50 states + DC)")

In [ ]:
# ---------------------------------------------------------------------------
# Inspect cleaned output
# ---------------------------------------------------------------------------

# Winner distribution and electoral vote totals
trump_wins = fec[fec["winner"] == "Trump"]
harris_wins = fec[fec["winner"] == "Harris"]
trump_ev = fec["electoral_votes_trump"].sum()
harris_ev = fec["electoral_votes_harris"].sum()

print(f"Trump wins:  {len(trump_wins)} states, {trump_ev} electoral votes")
print(f"Harris wins: {len(harris_wins)} states, {harris_ev} electoral votes")

# Vote share sanity check
share_sum = fec["trump_vote_share"] + fec["harris_vote_share"] + fec["other_vote_share"]
print(f"\nVote share sum check:")
print(f"  Mean: {share_sum.mean():.4f}")
print(f"  Min:  {share_sum.min():.4f}")
print(f"  Max:  {share_sum.max():.4f}")

# Swing state results
print(f"\nSwing state results:")
swing = fec[fec["is_swing"]].sort_values("state")
display(swing[["state", "state_name", "trump_vote_share", "harris_vote_share",
               "margin", "winner", "electoral_votes"]])

### FEC cleaning summary

The raw Excel file was filtered to the 50 states plus DC, removing territories
and summary rows. Column names were standardized to snake_case, vote and
electoral-vote columns cast to integers (NaN → 0), and derived columns added:
vote shares for each candidate, margin (positive = Trump lead), and a binary
`winner` column.

---

## Section 4: Events

The events timeline is a manually curated CSV listing major campaign events
during the 2024 election cycle. Each row records the date, a description, a
category (e.g., debate, convention), and which candidate the event primarily
affected.

**Cleaning challenges:**
- Trailing empty rows from manual editing
- Whitespace in string columns
- Need to derive `days_before_election` and boolean `affects_trump`/`affects_harris` flags

In [ ]:
# ---------------------------------------------------------------------------
# Assert raw file exists and inspect raw data
# ---------------------------------------------------------------------------

raw_events_path = EVENTS_RAW / "events.csv"
assert raw_events_path.exists(), f"Missing: {raw_events_path}"

raw_events = pd.read_csv(raw_events_path)
print(f"Shape: {raw_events.shape}")
print(f"Columns: {list(raw_events.columns)}")
print(f"\nCategories: {sorted(raw_events['category'].dropna().unique())}")
print(f"Affects values: {sorted(raw_events['affects'].dropna().unique())}")

display(raw_events)

In [ ]:
# ---------------------------------------------------------------------------
# Run the cleaning function
# ---------------------------------------------------------------------------

raw_events_rows = len(raw_events)

clean_events()

events = pd.read_csv(PROCESSED_DIR / "events.csv", parse_dates=["date"])

print(f"\nBefore/after:")
print(f"  Raw:     {raw_events_rows} rows")
print(f"  Cleaned: {len(events)} rows")

In [ ]:
# ---------------------------------------------------------------------------
# Inspect cleaned output: derived columns and full table
# ---------------------------------------------------------------------------

print(f"Date range: {events['date'].min().date()} to {events['date'].max().date()}")
print(f"Events affecting Trump:  {events['affects_trump'].sum()}")
print(f"Events affecting Harris: {events['affects_harris'].sum()}")
print(f"Derived columns: days_before_election, affects_trump, affects_harris")

display(events)

### Events cleaning summary

Trailing empty rows were dropped, whitespace stripped from all string columns,
and dates parsed. Three derived columns were added: `days_before_election`
(integer days from the event to November 5, 2024) and boolean flags
`affects_trump` and `affects_harris` (derived from the `affects` column,
which can be `"Trump"`, `"Harris"`, or `"both"`).

---

## Cross-Dataset Summary

In [ ]:
# ---------------------------------------------------------------------------
# Summary table of all cleaned datasets
# ---------------------------------------------------------------------------

pm_hourly = pd.read_csv(PROCESSED_DIR / "polymarket_hourly.csv")
pm_minutely = pd.read_csv(PROCESSED_DIR / "polymarket_minutely.csv")

summary_data = {
    "Dataset": [
        "Polymarket (daily)",
        "Polymarket (hourly)",
        "Polymarket (minutely)",
        "FiveThirtyEight",
        "FEC Results",
        "Events",
    ],
    "Rows": [
        len(pm_daily),
        len(pm_hourly),
        len(pm_minutely),
        len(p538),
        len(fec),
        len(events),
    ],
    "Columns": [
        pm_daily.shape[1],
        pm_hourly.shape[1],
        pm_minutely.shape[1],
        p538.shape[1],
        fec.shape[1],
        events.shape[1],
    ],
    "Granularity": [
        "State × day",
        "State × hour",
        "State × minute",
        "State × day",
        "State",
        "Event",
    ],
}

summary = pd.DataFrame(summary_data)
display(summary)

## Conclusions

### Data quality

All four raw sources required cleaning: file consolidation
(Polymarket), cycle filtering and format pivoting (FiveThirtyEight), territory
removal and type casting (FEC), and whitespace/empty-row handling (Events).
Post-cleaning sanity checks confirm that Polymarket probabilities sum to ~1.0,
FEC vote shares sum to ~1.0, and electoral vote totals match the certified
outcome.

### Key design decisions

1. **Biden–Harris merge**: FiveThirtyEight's Biden and Harris polling averages
   are combined into a single Democratic series with a `dem_candidate` attribute,
   enabling continuous time-series analysis across the candidate switch.
2. **Swing state tagging**: All datasets include an `is_swing` flag for the
   seven battleground states (AZ, GA, MI, NV, NC, PA, WI), enabling us to filter across analyses.
3. **Deterministic output**: Cleaning functions sort by state and date and
   resolve duplicates deterministically (latest timestamp chosen), ensuring
   identical output on repeated runs.

### Output files

The six cleaned CSVs in `data/processed/` serve as input to the analysis
notebooks:
- [Accuracy Analysis](./02_accuracy.ipynb) uses `polymarket_daily.csv`,
  `polls_538.csv`, and `fec_results.csv`
- [Event Response Analysis](./03_events.ipynb) uses `polymarket_daily.csv`,
  `polls_538.csv`, and `events.csv`